# Gemma Graph Pipeline

Deterministic staged notebook for building a grounded graph from a bounded document.

Flow:
1. Read document
2. Extract document overview + topic skeleton
3. Expand concepts per topic
4. Normalize and compress concepts
5. Infer prerequisite links
6. Merge into one Pydantic-validated graph


In [37]:
from __future__ import annotations

import json
import logging
import os
import re
import uuid
from pathlib import Path

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "config.py").exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from pydantic import BaseModel, Field
from pypdf import PdfReader

from core.config import APP_NAME, USER_ID


In [38]:
GOOGLE_GEMINI = (
    os.getenv("GOOGLE_GEMINI_API_KEY")
    or os.getenv("GOOGLE_GEMINI")
    or os.getenv("GEMINI_API_KEY")
)

if not GOOGLE_GEMINI:
    raise RuntimeError("Missing GOOGLE_GEMINI / GOOGLE_GEMINI_API_KEY in environment.")

MODEL_NAME = "gemini/gemma-4-31b-it"
MODEL = LiteLlm(model=MODEL_NAME, api_key=GOOGLE_GEMINI)


In [39]:
class Evidence(BaseModel):
    page: int | None = Field(
        default=None,
        description="1-based page number in the source document where this evidence appears, when available.",
    )
    section: str | None = Field(
        default=None,
        description="Section or heading label from the source document for this evidence, when available.",
    )
    excerpt: str = Field(
        min_length=1,
        description="A short verbatim excerpt from the source that grounds the concept.",
    )


class Topic(BaseModel):
    id: str = Field(
        min_length=1,
        description="Stable slug identifier for this topic, assigned in Python code.",
    )
    title: str = Field(
        min_length=1,
        description="Human-readable topic title taken from the material at medium granularity.",
    )
    summary: str = Field(
        min_length=1,
        description="Short explanation of what this topic is about.",
    )
    context: str = Field(
        min_length=1,
        description="How this topic functions in this specific document or why it matters here.",
    )


class Concept(BaseModel):
    id: str = Field(
        min_length=1,
        description="Stable slug identifier for this concept, assigned in Python code.",
    )
    topic_id: str = Field(
        min_length=1,
        description="Identifier of the topic this concept belongs to.",
    )
    title: str = Field(
        min_length=1,
        description="Human-readable concept title grounded in the source material.",
    )
    summary: str = Field(
        min_length=1,
        description="Short explanation of what the concept means.",
    )
    context: str = Field(
        min_length=1,
        description="How the concept is used, framed, or emphasized in this document.",
    )
    prerequisite_ids: list[str] = Field(
        default_factory=list,
        description="Concept ids that should usually be understood before this concept.",
    )
    evidence: list[Evidence] = Field(
        default_factory=list,
        description="One or more grounded evidence snippets that support this concept.",
    )


class DocumentNode(BaseModel):
    id: str = Field(
        min_length=1,
        description="Stable slug identifier for the source document.",
    )
    title: str = Field(
        min_length=1,
        description="Human-readable document title.",
    )
    source_type: str = Field(
        min_length=1,
        description="Source format such as pdf, txt, or md.",
    )
    domain: str = Field(
        min_length=1,
        description="Short phrase naming the domain or subject area of the material.",
    )
    overview: str = Field(
        min_length=1,
        description="Short overview explaining what the document covers and why it matters.",
    )


class StructuredGraph(BaseModel):
    document: DocumentNode = Field(
        description="Top-level metadata describing the source document.",
    )
    topics: list[Topic] = Field(
        description="Medium-granularity topic clusters extracted from the document.",
    )
    concepts: list[Concept] = Field(
        description="Grounded concepts extracted from the document and attached to topics.",
    )


class TopicDraft(BaseModel):
    title: str = Field(
        min_length=1,
        description="Human-readable title for one topic cluster from the material.",
    )
    summary: str = Field(
        min_length=1,
        description="Short explanation of what this topic is about.",
    )
    context: str = Field(
        min_length=1,
        description="How this topic functions in this specific document.",
    )


class DocumentScaffold(BaseModel):
    domain: str = Field(
        min_length=1,
        description="Short phrase naming the domain or subject area of the document.",
    )
    overview: str = Field(
        min_length=1,
        description="Short overview explaining the document's scope and importance.",
    )
    topics: list[TopicDraft] = Field(
        description="Initial topic scaffold extracted from the document before concept expansion.",
    )


class ConceptDraft(BaseModel):
    title: str = Field(
        min_length=1,
        description="Human-readable title for one grounded concept within the current topic.",
    )
    summary: str = Field(
        min_length=1,
        description="Short explanation of what the concept means.",
    )
    context: str = Field(
        min_length=1,
        description="How the concept is used, framed, or emphasized in this document.",
    )
    evidence: list[Evidence] = Field(
        default_factory=list,
        description="One or two grounded evidence snippets supporting this concept.",
    )


class TopicConceptBatch(BaseModel):
    concepts: list[ConceptDraft] = Field(
        description="Concepts extracted for one topic during the topic expansion stage.",
    )


class NormalizedConceptDraft(BaseModel):
    id: str = Field(
        min_length=1,
        description="Existing concept id to keep as the canonical identifier after normalization.",
    )
    title: str = Field(
        min_length=1,
        description="Refined concept title after deduplication and cleanup.",
    )
    summary: str = Field(
        min_length=1,
        description="Refined short explanation of the concept.",
    )
    context: str = Field(
        min_length=1,
        description="Refined explanation of how this concept functions in the document.",
    )
    evidence: list[Evidence] = Field(
        default_factory=list,
        description="Grounded evidence snippets preserved or improved during normalization.",
    )


class NormalizedConceptSet(BaseModel):
    concepts: list[NormalizedConceptDraft] = Field(
        description="Normalized concept set after removing duplicates or weak concepts.",
    )


class PrerequisiteLink(BaseModel):
    concept_id: str = Field(
        min_length=1,
        description="Target concept id whose prerequisite concepts are being listed.",
    )
    prerequisite_ids: list[str] = Field(
        default_factory=list,
        description="Concept ids that should usually be understood before the target concept.",
    )


class PrerequisiteLinkSet(BaseModel):
    links: list[PrerequisiteLink] = Field(
        description="Prerequisite links for the normalized concept set.",
    )


In [40]:
logging.getLogger("pypdf").setLevel(logging.ERROR)

MAX_EVIDENCE_PER_CONCEPT = 2


def slugify(value: str) -> str:
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-") or "node"


def make_unique_slug(title: str, used: set[str], prefix: str | None = None) -> str:
    base = slugify(title)
    candidate = f"{prefix}-{base}" if prefix else base
    if candidate not in used:
        used.add(candidate)
        return candidate

    index = 2
    while True:
        next_candidate = f"{candidate}-{index}"
        if next_candidate not in used:
            used.add(next_candidate)
            return next_candidate
        index += 1


def log_progress(message: str, verbose: bool = True) -> None:
    if verbose:
        print(message, flush=True)


def clean_text(value: str | None) -> str:
    if not value:
        return ""
    return re.sub(r"\s+", " ", value).strip()


def compact_json(value: object) -> str:
    return json.dumps(value, ensure_ascii=False, indent=2)


def compact_topic_payload(topic: Topic) -> dict[str, str]:
    return {
        "id": topic.id,
        "title": topic.title,
        "summary": topic.summary,
        "context": topic.context,
    }


def compact_concept_payload(concept: Concept) -> dict[str, str]:
    return {
        "id": concept.id,
        "topic_id": concept.topic_id,
        "title": concept.title,
        "summary": concept.summary,
        "context": concept.context,
    }


def clean_evidence_list(
    evidence: list[Evidence],
    limit: int = MAX_EVIDENCE_PER_CONCEPT,
) -> list[Evidence]:
    cleaned: list[Evidence] = []
    seen: set[tuple[int | None, str | None, str]] = set()
    for item in evidence:
        excerpt = clean_text(item.excerpt)
        if not excerpt:
            continue
        section = clean_text(item.section) or None
        key = (item.page, section, excerpt)
        if key in seen:
            continue
        cleaned.append(Evidence(page=item.page, section=section, excerpt=excerpt))
        seen.add(key)
        if len(cleaned) >= limit:
            break
    return cleaned


def clean_concept_fields(title: str, summary: str, context: str) -> tuple[str, str, str]:
    return clean_text(title), clean_text(summary), clean_text(context)


def build_concept_from_draft(concept_id: str, topic_id: str, draft: ConceptDraft) -> Concept:
    title, summary, context = clean_concept_fields(draft.title, draft.summary, draft.context)
    return Concept(
        id=concept_id,
        topic_id=topic_id,
        title=title,
        summary=summary,
        context=context,
        prerequisite_ids=[],
        evidence=clean_evidence_list(draft.evidence),
    )


def dedupe_concepts(concepts: list[Concept], verbose: bool = True) -> list[Concept]:
    deduped: list[Concept] = []
    seen_keys: set[tuple[str, str]] = set()
    for concept in concepts:
        key = (concept.topic_id, slugify(concept.title))
        if key in seen_keys:
            log_progress(f"[dedupe] dropped exact duplicate concept: {concept.id}", verbose)
            continue
        seen_keys.add(key)
        deduped.append(concept)
    return deduped


def has_path(graph: dict[str, list[str]], start: str, target: str, seen: set[str] | None = None) -> bool:
    if start == target:
        return True
    if seen is None:
        seen = set()
    if start in seen:
        return False
    seen.add(start)
    for nxt in graph.get(start, []):
        if has_path(graph, nxt, target, seen):
            return True
    return False


def prune_prerequisite_cycles(graph: dict[str, list[str]]) -> dict[str, list[str]]:
    pruned: dict[str, list[str]] = {node: [] for node in graph}
    for node, deps in graph.items():
        for dep in deps:
            if dep == node:
                continue
            if has_path(pruned, dep, node):
                continue
            pruned[node].append(dep)
    return pruned


def read_document(path: str, max_pages: int = 10, max_chars: int = 24000) -> dict[str, str]:
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        reader = PdfReader(str(file_path))
        pages = []
        for page in reader.pages[:max_pages]:
            pages.append(page.extract_text() or "")
        text = "\n\n".join(pages)
        source_type = "pdf"
    elif suffix in {".txt", ".md"}:
        text = file_path.read_text(encoding="utf-8")
        source_type = suffix.lstrip(".")
    else:
        raise ValueError("Supported file types in this notebook: .pdf, .txt, .md")

    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if not text:
        raise ValueError(f"No text extracted from {file_path.name}")

    return {
        "document_id": slugify(file_path.stem),
        "title": clean_text(file_path.stem.replace("_", " ").replace("-", " ").strip() or file_path.stem),
        "source_type": source_type,
        "text": text[:max_chars],
    }


async def run_structured(agent: Agent, prompt: str, app_suffix: str) -> str:
    runner = Runner(
        agent=agent,
        app_name=f"{APP_NAME}_{app_suffix}",
        session_service=InMemorySessionService(),
    )
    session_id = str(uuid.uuid4())
    await runner.session_service.create_session(
        app_name=runner.app_name,
        user_id=USER_ID,
        session_id=session_id,
    )

    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    final_text = None
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                part.text
                for part in event.content.parts
                if getattr(part, "text", None) and not getattr(part, "thought", False)
            ]
            final_text = "\n".join(texts).strip()

    if not final_text:
        raise RuntimeError(f"No final output received for {app_suffix}.")

    return final_text


In [41]:
SCAFFOLD_PROMPT = """
You are building a minimal grounded graph scaffold from a bounded document.

Return valid JSON matching the DocumentScaffold schema exactly.
Do not include markdown fences or commentary.

Rules:
- Identify the domain of the material.
- Write a short overview that explains what the material covers and why it matters.
- Produce 2-6 topics.
- Topics must be medium-granularity clusters from the material.
- Each topic needs:
  - title
  - summary: what the topic is
  - context: how that topic functions in this specific document
- Do not include concepts yet.
- Keep the output minimal, grounded, and specific to the document.
"""

TOPIC_CONCEPT_PROMPT = """
You expand one topic into a small set of grounded concepts.

Return valid JSON matching the TopicConceptBatch schema exactly.
Do not include markdown fences or commentary.

Rules:
- Work only inside the provided topic.
- Produce 1-5 concepts for this topic.
- Each concept needs:
  - title
  - summary: what the concept means
  - context: how the concept is used or framed in this document
  - evidence: 1-2 short grounded excerpts
- You will also receive already created concepts from earlier topics.
- Do not duplicate or trivially rename an already created concept.
- If an idea is already covered by an existing concept, skip it unless this topic adds a clearly distinct role or framing.
- Prefer the document's own named terms and phrases over abstract paraphrases when possible.
- Keep a concept in the topic that most directly frames or explains it in the source.
- Do not pull a concept into the current topic just because it is mentioned here if another topic is its clearer home.
- Prefer concepts that are more specific than the topic title itself.
- Avoid returning a concept whose title substantially repeats the topic title unless the source clearly presents it as a distinct concept.
- Do not invent concepts unsupported by the source.
- Do not include ids or prerequisite links.
- Keep concepts atomic and useful for later tutoring/planning.
- It is acceptable to return fewer concepts when the topic is narrow.
"""

NORMALIZER_PROMPT = """
You normalize an extracted concept set before prerequisite linking.

Return valid JSON matching the NormalizedConceptSet schema exactly.
Do not include markdown fences or commentary.

Rules:
- Use only the provided concept ids.
- Each returned concept keeps one existing concept id as its canonical id.
- You may omit concepts to drop them.
- Do not invent new concept ids or new topics.
- Remove or compress concepts that are duplicates, trivial restatements, or too vague to be useful.
- Prefer the document's own named concepts and phrases over abstract rewordings.
- Preserve contrastive or named source concepts when they are explicitly presented in the document.
- Do not collapse a concept into a generic abstraction if the source uses a more specific term.
- Keep each concept in the topic that best matches how the source frames it.
- Prefer distinct concepts that add explanatory value for later tutoring and planning.
- Keep concepts grounded in the document and preserve or improve evidence quality.
- Do not rewrite the whole graph from scratch. This is a cleanup pass over the extracted concepts.
"""

LINKER_PROMPT = """
You infer prerequisite links between already extracted concepts.

Return valid JSON matching the PrerequisiteLinkSet schema exactly.
Do not include markdown fences or commentary.

Rules:
- Use only the provided concept ids.
- Add a prerequisite only when understanding one concept clearly helps explain another.
- Be conservative: if the dependency is weak, debatable, or only loosely related, leave it empty.
- Prefer explicit conceptual prerequisites over broad thematic associations.
- Do not create a prerequisite just because one concept motivates, contrasts with, or follows another in the document.
- Keep links sparse and useful.
- Do not create self-links.
- Do not invent unknown ids.
- It is acceptable for many concepts to have empty prerequisite lists.
"""

scaffold_agent = Agent(
    model=MODEL,
    name="GraphScaffold",
    instruction=SCAFFOLD_PROMPT,
    output_schema=DocumentScaffold,
    generate_content_config=types.GenerateContentConfig(temperature=0),
)

topic_concept_agent = Agent(
    model=MODEL,
    name="TopicConceptExpander",
    instruction=TOPIC_CONCEPT_PROMPT,
    output_schema=TopicConceptBatch,
    generate_content_config=types.GenerateContentConfig(temperature=0),
)

normalizer_agent = Agent(
    model=MODEL,
    name="ConceptNormalizer",
    instruction=NORMALIZER_PROMPT,
    output_schema=NormalizedConceptSet,
    generate_content_config=types.GenerateContentConfig(temperature=0),
)

linker_agent = Agent(
    model=MODEL,
    name="PrerequisiteLinker",
    instruction=LINKER_PROMPT,
    output_schema=PrerequisiteLinkSet,
    generate_content_config=types.GenerateContentConfig(temperature=0),
)


In [42]:
async def extract_scaffold(
    payload: dict[str, str],
    verbose: bool = True,
) -> tuple[DocumentNode, list[Topic]]:
    log_progress(f"[read] {payload['title']} ({payload['source_type']})", verbose)
    log_progress("[scaffold] extracting document overview and topic skeleton...", verbose)
    prompt = (
        "Document metadata:\n"
        + compact_json(
            {
                "id": payload["document_id"],
                "title": payload["title"],
                "source_type": payload["source_type"],
            }
        )
        + "\n\nDocument text:\n"
        + payload["text"]
    )
    raw = await run_structured(scaffold_agent, prompt, "GRAPH_SCAFFOLD")
    scaffold = DocumentScaffold.model_validate_json(raw)

    used_topic_ids: set[str] = set()
    topics = [
        Topic(
            id=make_unique_slug(topic.title, used_topic_ids),
            title=clean_text(topic.title),
            summary=clean_text(topic.summary),
            context=clean_text(topic.context),
        )
        for topic in scaffold.topics
    ]

    document = DocumentNode(
        id=payload["document_id"],
        title=payload["title"],
        source_type=payload["source_type"],
        domain=clean_text(scaffold.domain),
        overview=clean_text(scaffold.overview),
    )

    log_progress(f"[scaffold] domain: {document.domain}", verbose)
    log_progress(f"[scaffold] overview: {document.overview}", verbose)
    for index, topic in enumerate(topics, start=1):
        log_progress(
            f"[topic {index}/{len(topics)}] {topic.id} -> {topic.title}",
            verbose,
        )
    return document, topics


async def extract_concepts_for_topic(
    payload: dict[str, str],
    document: DocumentNode,
    topics: list[Topic],
    topic: Topic,
    existing_concepts: list[Concept],
    used_concept_ids: set[str],
    verbose: bool = True,
) -> list[Concept]:
    existing_concepts_payload = [compact_concept_payload(concept) for concept in existing_concepts]

    log_progress(f"[concepts] expanding {topic.title}...", verbose)
    prompt = (
        "Document:\n"
        + compact_json(document.model_dump())
        + "\n\nAll topics:\n"
        + compact_json([compact_topic_payload(item) for item in topics])
        + "\n\nCurrent topic:\n"
        + compact_json(compact_topic_payload(topic))
        + "\n\nAlready created concepts:\n"
        + compact_json(existing_concepts_payload)
        + "\n\nDocument text:\n"
        + payload["text"]
    )
    raw = await run_structured(topic_concept_agent, prompt, f"GRAPH_TOPIC_{topic.id}")
    batch = TopicConceptBatch.model_validate_json(raw)

    concepts: list[Concept] = []
    for draft in batch.concepts:
        concept_id = make_unique_slug(draft.title, used_concept_ids, prefix=topic.id)
        concepts.append(build_concept_from_draft(concept_id, topic.id, draft))

    concepts = dedupe_concepts(concepts, verbose=verbose)
    if concepts:
        for concept in concepts:
            log_progress(f"  + {concept.id}: {concept.title}", verbose)
    else:
        log_progress("  + no new concepts returned", verbose)
    return concepts


async def normalize_concepts(
    document: DocumentNode,
    topics: list[Topic],
    concepts: list[Concept],
    verbose: bool = True,
) -> list[Concept]:
    if not concepts:
        return concepts

    log_progress("[normalize] compressing extracted concepts...", verbose)
    prompt = (
        "Document:\n"
        + compact_json(document.model_dump())
        + "\n\nTopics:\n"
        + compact_json([compact_topic_payload(topic) for topic in topics])
        + "\n\nExtracted concepts:\n"
        + compact_json([concept.model_dump() for concept in concepts])
    )
    raw = await run_structured(normalizer_agent, prompt, "GRAPH_NORMALIZER")
    normalized_set = NormalizedConceptSet.model_validate_json(raw)

    original_by_id = {concept.id: concept for concept in concepts}
    topic_by_id = {topic.id: topic for topic in topics}

    normalized: list[Concept] = []
    kept_ids: set[str] = set()
    for item in normalized_set.concepts:
        if item.id not in original_by_id:
            continue
        if item.id in kept_ids:
            continue
        original = original_by_id[item.id]
        title, summary, context = clean_concept_fields(item.title, item.summary, item.context)
        normalized.append(
            Concept(
                id=original.id,
                topic_id=original.topic_id,
                title=title,
                summary=summary,
                context=context,
                prerequisite_ids=[],
                evidence=clean_evidence_list(item.evidence),
            )
        )
        kept_ids.add(item.id)

    def concept_matches_topic(concept: Concept) -> bool:
        topic = topic_by_id[concept.topic_id]
        return slugify(concept.title) == slugify(topic.title)

    concepts_by_topic: dict[str, list[Concept]] = {}
    for concept in concepts:
        concepts_by_topic.setdefault(concept.topic_id, []).append(concept)

    normalized_topic_ids = {concept.topic_id for concept in normalized}
    for topic in topics:
        if topic.id in normalized_topic_ids:
            continue
        fallback_candidates = concepts_by_topic.get(topic.id, [])
        if not fallback_candidates:
            continue
        fallback = next(
            (concept for concept in fallback_candidates if not concept_matches_topic(concept)),
            fallback_candidates[0],
        )
        if fallback.id not in kept_ids:
            normalized.append(fallback)
            kept_ids.add(fallback.id)
            log_progress(
                f"  ! restored fallback concept for topic {topic.id}: {fallback.id}",
                verbose,
            )

    normalized = dedupe_concepts(normalized, verbose=verbose)
    normalized.sort(key=lambda concept: concepts.index(original_by_id[concept.id]))
    dropped_ids = [concept.id for concept in concepts if concept.id not in kept_ids]
    if dropped_ids:
        log_progress(f"[normalize] dropped: {dropped_ids}", verbose)
    for concept in normalized:
        log_progress(f"  = {concept.id}: {concept.title}", verbose)
    return normalized


async def infer_prerequisites(
    document: DocumentNode,
    topics: list[Topic],
    concepts: list[Concept],
    verbose: bool = True,
) -> dict[str, list[str]]:
    log_progress("[links] inferring prerequisite links...", verbose)
    prompt = (
        "Document:\n"
        + compact_json(document.model_dump())
        + "\n\nTopics:\n"
        + compact_json([compact_topic_payload(topic) for topic in topics])
        + "\n\nConcepts:\n"
        + compact_json([compact_concept_payload(concept) for concept in concepts])
    )
    raw = await run_structured(linker_agent, prompt, "GRAPH_LINKER")
    link_set = PrerequisiteLinkSet.model_validate_json(raw)

    valid_ids = {concept.id for concept in concepts}
    links_by_concept: dict[str, list[str]] = {concept.id: [] for concept in concepts}
    for link in link_set.links:
        if link.concept_id not in valid_ids:
            continue
        cleaned: list[str] = []
        for prereq_id in link.prerequisite_ids:
            if prereq_id == link.concept_id:
                continue
            if prereq_id not in valid_ids:
                continue
            if prereq_id in cleaned:
                continue
            cleaned.append(prereq_id)
        links_by_concept[link.concept_id] = cleaned

    links_by_concept = prune_prerequisite_cycles(links_by_concept)
    for concept in concepts:
        log_progress(f"  - {concept.id} <- {links_by_concept.get(concept.id, [])}", verbose)
    return links_by_concept


In [43]:
async def build_graph_pipeline(
    path: str,
    max_pages: int = 10,
    verbose: bool = True,
) -> StructuredGraph:
    payload = read_document(path, max_pages=max_pages)
    document, topics = await extract_scaffold(payload, verbose=verbose)

    used_concept_ids: set[str] = set()
    concepts: list[Concept] = []
    for topic in topics:
        new_concepts = await extract_concepts_for_topic(
            payload=payload,
            document=document,
            topics=topics,
            topic=topic,
            existing_concepts=concepts,
            used_concept_ids=used_concept_ids,
            verbose=verbose,
        )
        concepts.extend(new_concepts)

    concepts = dedupe_concepts(concepts, verbose=verbose)
    concepts = await normalize_concepts(
        document,
        topics,
        concepts,
        verbose=verbose,
    )

    prerequisite_map = await infer_prerequisites(
        document,
        topics,
        concepts,
        verbose=verbose,
    )
    final_concepts = [
        concept.model_copy(update={"prerequisite_ids": prerequisite_map.get(concept.id, [])})
        for concept in concepts
    ]

    log_progress(
        f"[done] built graph with {len(topics)} topics and {len(final_concepts)} concepts",
        verbose,
    )
    return StructuredGraph(
        document=document,
        topics=topics,
        concepts=final_concepts,
    )


In [ ]:
# Example:
graph = await build_graph_pipeline("../data/sample_docs/Agent Quality.pdf", max_pages=10)


[read] Agent Quality (pdf)
[scaffold] extracting document overview and topic skeleton...


In [ ]:
graph.model_dump()